In [ ]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.4/263.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import dagshub
import mlflow
dagshub.init(repo_owner='Aryanupadhyay23', repo_name='Twitter-Sentiment-Analysis-MLOps', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6d2db812-9193-42e4-ae72-137138a73465&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=9dd41206a2027c435ad150889755d51765f8beef37e12ae62359dd64cc4469c6




Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow")

In [ ]:
mlflow.set_experiment("Bow vs Tfidf")

<Experiment: artifact_location='mlflow-artifacts:/e3bf555e8cb0479989bb97383b20c4df', creation_time=1771667531300, experiment_id='2', last_update_time=1771667531300, lifecycle_stage='active', name='Bow vs Tfidf', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('twitter_cleaned.csv')

In [ ]:
df.head()

,sentiment,text
0,positive,come border kill
1,positive,im get borderland kill
2,positive,im come borderland murder
3,positive,im get borderland 2 murder
4,positive,im get borderland murder


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51615 entries, 0 to 51617
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  51615 non-null  object
 1   text       51615 non-null  object
dtypes: object(2)
memory usage: 1.2+ MB


In [ ]:
df.dropna(inplace=True)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
import json
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [ ]:
def run_experiment(df, vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):

    # Remove null values
    df = df.dropna(subset=["text", "sentiment"]).copy()
    df["text"] = df["text"].astype(str)

    with mlflow.start_run():

        # Set experiment tags
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_LogReg")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "LogisticRegression")
        mlflow.set_tag("vectorizer_name", vectorizer_name)

        # Create vectorizer
        if vectorizer_type == "BoW":
            vectorizer = CountVectorizer(ngram_range=ngram_range,
                                         max_features=vectorizer_max_features)
        else:
            vectorizer = TfidfVectorizer(ngram_range=ngram_range,
                                         max_features=vectorizer_max_features)

        # Transform text into features
        X = vectorizer.fit_transform(df["text"])
        y = df["sentiment"]

        # Log dataset info
        mlflow.log_param("vocab_size", len(vectorizer.vocabulary_))
        mlflow.log_param("total_samples", X.shape[0])
        mlflow.log_param("num_classes", len(np.unique(y)))

        # Log class distribution
        class_counts = y.value_counts().to_dict()
        for cls, count in class_counts.items():
            mlflow.log_param(f"class_count_{str(cls)}", int(count))

        # Perform stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

        mlflow.log_param("train_samples", X_train.shape[0])
        mlflow.log_param("test_samples", X_test.shape[0])

        # Define model hyperparameters
        C = 1.0
        penalty = "l2"
        solver = "lbfgs"
        max_iter = 1000

        mlflow.log_params({
            "vectorizer_type": vectorizer_type,
            "ngram_range": str(ngram_range),
            "max_features": vectorizer_max_features,
            "C": C,
            "penalty": penalty,
            "solver": solver,
            "max_iter": max_iter,
            "random_state": 42
        })

        # Initialize logistic regression
        model = LogisticRegression(C=C,
                                   penalty=penalty,
                                   solver=solver,
                                   max_iter=max_iter,
                                   random_state=42)

        # Train model
        model.fit(X_train, y_train)

        # Generate predictions
        y_pred = model.predict(X_test)

        # Compute overall metrics
        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average="macro")
        weighted_f1 = f1_score(y_test, y_pred, average="weighted")
        micro_f1 = f1_score(y_test, y_pred, average="micro")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("weighted_f1", weighted_f1)
        mlflow.log_metric("micro_f1", micro_f1)

        # Generate classification report
        report_dict = classification_report(y_test, y_pred, output_dict=True)

        # Log per-class metrics
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                mlflow.log_metric(f"class_{label}_precision", metrics.get("precision", 0))
                mlflow.log_metric(f"class_{label}_recall", metrics.get("recall", 0))
                mlflow.log_metric(f"class_{label}_f1", metrics.get("f1-score", 0))

        # Save classification report artifact
        with open("classification_report.json", "w") as f:
            json.dump(report_dict, f, indent=4)

        mlflow.log_artifact("classification_report.json")

        # Create confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(6, 5))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix - {vectorizer_name} {ngram_range}")
        plt.tight_layout()
        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        # Log model with signature
        signature = infer_signature(X_train[:100].toarray(),
                                    model.predict(X_train[:100]))

        mlflow.sklearn.log_model(model,
                                 artifact_path="model",
                                 signature=signature,
                                 input_example=X_train[:5])

        print(f"Run completed: {vectorizer_name}, ngram={ngram_range}")

In [ ]:
ngram_ranges = [(1, 1), (1, 2), (1, 3)]
max_features_list = [5000]
vectorizer_types = ["BoW", "TF-IDF"]

for max_features in max_features_list:
    for ngram_range in ngram_ranges:
        for vectorizer_type in vectorizer_types:

            run_experiment(
                df,
                vectorizer_type=vectorizer_type,
                ngram_range=ngram_range,
                vectorizer_max_features=max_features,
                vectorizer_name=vectorizer_type
            )

2026/02/21 10:24:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 10:25:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed: BoW, ngram=(1, 1)
🏃 View run BoW_(1, 1)_LogReg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2/runs/df9200442da1404582fa1cfdf6871385
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2


2026/02/21 10:26:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 10:26:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed: TF-IDF, ngram=(1, 1)
🏃 View run TF-IDF_(1, 1)_LogReg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2/runs/c98da5071ee847adb1a779e93584f5a4
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2


2026/02/21 10:27:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 10:27:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Run completed: TF-IDF, ngram=(1, 2)
🏃 View run TF-IDF_(1, 2)_LogReg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2/runs/b18a8a4b86fd472bb8e0512992b6d6b2
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2


2026/02/21 10:29:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 10:30:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed: BoW, ngram=(1, 3)
🏃 View run BoW_(1, 3)_LogReg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2/runs/7d006ce8f152496ba12944490bd8068a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2


2026/02/21 10:30:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 10:31:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed: TF-IDF, ngram=(1, 3)
🏃 View run TF-IDF_(1, 3)_LogReg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2/runs/1e2e8f3692b44e9a8bc495c28178cd62
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/2


In [ ]:
def tune_bow_feature_size_rich(df):

    feature_sizes = list(range(1000, 10001, 1000))

    best_score = 0
    best_feature_size = None

    mlflow.set_experiment("BoW_(1,2)_Feature_Tuning")

    for max_features in feature_sizes:

        with mlflow.start_run(run_name=f"BoW_(1,2)_features_{max_features}"):

            # Create vectorizer
            vectorizer = CountVectorizer(
                ngram_range=(1, 2),
                max_features=max_features
            )

            X = vectorizer.fit_transform(df["text"])
            y = df["sentiment"]

            # Log dataset information
            mlflow.log_param("vectorizer", "BoW")
            mlflow.log_param("ngram_range", "(1,2)")
            mlflow.log_param("max_features", max_features)
            mlflow.log_param("vocab_size", len(vectorizer.vocabulary_))
            mlflow.log_param("total_samples", X.shape[0])
            mlflow.log_param("num_classes", len(np.unique(y)))

            # Train-test split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=0.2,
                random_state=42,
                stratify=y
            )

            mlflow.log_param("train_samples", X_train.shape[0])
            mlflow.log_param("test_samples", X_test.shape[0])

            # Train model
            model = LogisticRegression(
                C=1.0,
                penalty="l2",
                solver="lbfgs",
                max_iter=1000,
                random_state=42
            )

            model.fit(X_train, y_train)

            # Predictions
            y_pred = model.predict(X_test)

            # Overall metrics
            accuracy = accuracy_score(y_test, y_pred)
            macro_f1 = f1_score(y_test, y_pred, average="macro")
            weighted_f1 = f1_score(y_test, y_pred, average="weighted")
            micro_f1 = f1_score(y_test, y_pred, average="micro")

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("macro_f1", macro_f1)
            mlflow.log_metric("weighted_f1", weighted_f1)
            mlflow.log_metric("micro_f1", micro_f1)

            # Per-class metrics
            report_dict = classification_report(y_test, y_pred, output_dict=True)

            for label, metrics in report_dict.items():
                if isinstance(metrics, dict):
                    mlflow.log_metric(f"class_{label}_precision", metrics.get("precision", 0))
                    mlflow.log_metric(f"class_{label}_recall", metrics.get("recall", 0))
                    mlflow.log_metric(f"class_{label}_f1", metrics.get("f1-score", 0))

            # Save classification report
            with open("classification_report.json", "w") as f:
                json.dump(report_dict, f, indent=4)

            mlflow.log_artifact("classification_report.json")

            # Confusion matrix
            conf_matrix = confusion_matrix(y_test, y_pred)

            plt.figure(figsize=(6, 5))
            sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.title(f"Confusion Matrix - BoW (1,2) - {max_features}")
            plt.tight_layout()
            plt.savefig("confusion_matrix.png")
            plt.close()

            mlflow.log_artifact("confusion_matrix.png")

            # Track best macro F1
            if macro_f1 > best_score:
                best_score = macro_f1
                best_feature_size = max_features

    print(f"\nBest max_features: {best_feature_size}")
    print(f"Best macro_f1: {best_score:.4f}")

    return best_feature_size, best_score

In [ ]:
best_features, best_score = tune_bow_feature_size_rich(df)

2026/02/21 10:44:24 INFO mlflow.tracking.fluent: Experiment with name 'BoW_(1,2)_Feature_Tuning' does not exist. Creating a new experiment.


🏃 View run BoW_(1,2)_features_1000 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4/runs/561f77d8317c446594d060dfe030c0fb
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4
🏃 View run BoW_(1,2)_features_2000 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4/runs/44169759b11443479cb4089d5268b183
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4
🏃 View run BoW_(1,2)_features_3000 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4/runs/c8fb607249c3455a9c27044d6d9946f9
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4
🏃 View run BoW_(1,2)_features_4000 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/4/runs/8f6cc78ef100425